# AC Optimal Power Flow (AC-OPF) + Discrete Reactive Power Compensation

Transmission system operators (ISOs/RTOs and dispatch centers) must determine the active/reactive output of each generator and the voltage at each bus to minimize generation costs while meeting demand. **DC approximation** (active power only, linear) is sufficient for congestion management, but voltage violations, reactive power shortages, and transmission losses **can only be seen through true alternating current (AC) power flow calculations**. This notebook provides a contrast to other samples (which have a big-M integer × continuous structure) and deals with a MINLP that includes **true non-convexities** (voltage × voltage × trigonometric functions) for the first time.

The net active/reactive injection at each bus $i$ is a **true non-convex function** of voltage magnitude $V$ and phase angle $\theta$:

$$
\begin{aligned}
P_i(V,\theta) &= V_i \sum_j V_j\,(G_{ij}\cos(\theta_i-\theta_j) + B_{ij}\sin(\theta_i-\theta_j)),\\
Q_i(V,\theta) &= V_i \sum_j V_j\,(G_{ij}\sin(\theta_i-\theta_j) - B_{ij}\cos(\theta_i-\theta_j)).
\end{aligned}
$$

The number of engaged steps of discrete capacitor banks $n_i \in \{0,\dots,5\}$ (integer) provides reactive power compensation — this is the core of making this model a MINLP, but since it enters the reactive balance **linearly**, the true non-convexity is concentrated on the $V\cdot V\cdot\cos/\sin$ side (multiplying integers directly into non-convex terms, such as in transmission line on/off or discrete taps, is more difficult and represents a future extension).

1. **Naive Formulation** — Polar coordinate format, epigraphized quadratic generation costs, usage of sin/cos.
2. **Diagnosis** — Observe with `mk.analyze` to see what symptoms true non-convexity produces (contrast with other big-M samples).
3. **Looking Inside the Diagnosis** — Look directly at root LP relaxation violations and the spatial branch-and-bound tree.
4. **Trying Improvements** — Because of the true non-convexity, try improvements (SCIP heuristic settings) focused on the "quality of feasible solutions" rather than "optimality proof", and measure the effects honestly.

Subject: `samples/energy_and_microgrid/ac_opf.py`

In [ ]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/energy_and_microgrid"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from pyscipopt import Model, quicksum, SCIP_PARAMSETTING

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import ac_opf
from minlpkit.collectors.violation import collect_root_violations, violation_by_type
from minlpkit.collectors.tree import solve_and_collect

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Naive Formulation

`build_model("small")` is for 5 buses (to check feasibility, but still truly non-convex).
The generation cost is quadratic convex (`a+b*Pg+c*Pg^2`), but since **SCIP cannot handle non-linear objectives directly**, it is moved to an epigraph variable `cost_g >= a+b*Pg+c*Pg^2` — a standard practice for MINLPs with non-linear objectives.

In [ ]:
m = ac_opf.build_model("small")
nB, nEdge = m.data["dims"]
n_int = sum(1 for v in m.getVars() if v.vtype() in ("INTEGER", "BINARY"))
n_nonlin = sum(1 for c in m.getConss() if not c.isLinear())
print(f"バス数={nB} 送電線数={nEdge} 発電機バス={len(m.data['gen_buses'])} "
      f"コンデンサ候補バス={len(m.data['cap_buses'])}")
print(f"変数: 整数(コンデンサ段数) {n_int} 個 / 連続(V, theta, Pg, Qg, cost) "
      f"{m.getNVars() - n_int} 個")
print(f"制約: {m.getNConss()} 本(内 非線形 = {n_nonlin} 本 = 有効/無効バランス各バス分)")
print("目的: epigraph 'cost_g >= a + b*Pg + c*Pg^2'(2次凸を線形不等式の下界として表現)")

## 2. Diagnosis (`mk.analyze`)

Unlike other samples with integer $\times$ continuous products or disjunctive structures, this problem has true non-convexity in the form of **voltage $\times$ voltage $\times$ trigonometric functions**. Let's see what symptoms appear at the small scale.

In [ ]:
report_small = mk.analyze(lambda: ac_opf.build_model("small"), name="acopf-small", time_limit=35)
print(report_small.summary())
print()
print({k: report_small.metrics.get(k) for k in [
    "gap", "nodes", "nsols", "spatial_share", "n_stalls", "has_nonlinear",
    "bottleneck_type", "bottleneck_rel_viol", "max_linking_groups", "n_heavy_linking"]})

All other big-M samples were solved at `nodes=1` (root) — here, **both `weak_relaxation` (concentrated relaxation violations + numerous spatial branches) and `dual_stall` (dual bound stagnation) trigger**. This is the qualitative difference between "true non-convexity" and "big-M disjunctive". For reference, let's see what happens at the default scale (14 buses, IEEE 14 equivalent) with the same time limit.

In [ ]:
report_default = mk.analyze(lambda: ac_opf.build_model("default"), name="acopf-default", time_limit=30)
print({k: report_default.metrics.get(k) for k in ["gap", "nodes", "nsols", "spatial_share"]})
print(f"gap {report_default.metrics.get('gap', 0)*100:.0f}% (30秒、実行可能解"
      f"{report_default.metrics.get('nsols')}個) — バス数が3倍弱になっただけでgapは"
      f"桁違いに悪化する。真の非凸の空間分枝限定法はスケールに極めて敏感。")

## 3. Looking Inside the Diagnosis

### 3-1. Root LP Relaxation Violations — Where is the dominant bottleneck?

In [ ]:
m3 = ac_opf.build_model("small")
vdf = collect_root_violations(m3)
by_type = violation_by_type(vdf)
print(by_type.to_string(index=False))

In [ ]:
top_rows = vdf.sort_values("rel_violation", ascending=False).head(nB)
fig = go.Figure(layout=base_layout(
    "ルートLP緩和での非線形制約 違反量(相対) — pbal(有効電力バランス)が支配的",
    "制約", "相対違反", height=380))
fig.add_trace(go.Bar(x=top_rows["constraint"], y=top_rows["rel_violation"],
    marker_color=[C["warn"] if t.startswith("pbal") else C["s1"] for t in top_rows["ctype"]],
    text=[f"{v:.2f}" for v in top_rows["rel_violation"]], textposition="outside", cliponaxis=False))
show(fig)

Under linearization (McCormick-like) relaxation, the violation of the active power balance `pbal` exceeds 60% — meaning that the linear relaxation hardly captures the true bilinear + trigonometric product of voltage $\times$ voltage $\times$ cos/sin. The looseness of the relaxation is orders of magnitude different from the integer $\times$ continuous products (McCormick relaxation) seen in other samples.

### 3-2. Spatial Branch-and-Bound Tree — Where do branches concentrate?

Collect branch nodes with `solve_and_collect` and color-code by the branch variable type (spatial = spatial branching on continuous variables / integer = capacitor steps).

In [ ]:
m4 = ac_opf.build_model("small")
tree_df = solve_and_collect(m4, max_nodes=1200, node_limit=1200)
counts = tree_df["kind"].value_counts()
print(counts)
print(f"最大深さ: {tree_df['depth'].max()}")

In [ ]:
kind_colors = {"spatial": C["s1"], "integer": C["warn"], "root": C["muted"]}
fig = go.Figure(layout=base_layout(
    "空間分枝木(先頭1200ノード) — spatial(電圧・位相角)が分枝の大半を占める",
    "tidy木レイアウト(x)", "深さ(下ほど深い)", height=440))
for kind, color in kind_colors.items():
    sub = tree_df[tree_df["kind"] == kind]
    if sub.empty:
        continue
    fig.add_trace(go.Scattergl(x=sub["x"], y=sub["y"], mode="markers", name=kind,
        marker=dict(size=4, color=color, opacity=0.7),
        hovertemplate="node%{text}<extra></extra>", text=sub["node"]))
show(fig)

In [ ]:
spatial_vars = tree_df.loc[tree_df["kind"] == "spatial", "branch_var"].value_counts().head(8)
integer_vars = tree_df.loc[tree_df["kind"] == "integer", "branch_var"].value_counts().head(8)
print("空間分枝の変数トップ8:")
print(spatial_vars.to_string())
print("整数分枝の変数トップ8:")
print(integer_vars.to_string() if not integer_vars.empty else "(整数分枝なし)")

Spatial branches are concentrated on voltages `V_*` and phase angles `theta_*` — exactly the textbook behavior of McCormick/spatial branch-and-bound methods splitting continuous variables to tighten the relaxation of non-convex power flows. There are very few integer branches (capacitor steps) — because this model is designed so integer decisions enter the reactive balance linearly (see ac_opf.py docstring), the true difficulty is concentrated on the continuous side.

## 4. Trying Improvements — Focusing on "Quality of Feasible Solutions" rather than "Optimality Proof"

In a truly non-convex MINLP, even at this scale (5 buses), a gap of nearly 200% remains after 30 seconds — **improvements focused on proving strict optimality are unrealistic**. What is practically important is "how good a feasible solution (generation cost) can be obtained within a limited time." We will change SCIP's heuristic intensity (`setHeuristics`) and compare the quality and number of primal solutions under the same time budget.

In [ ]:
def measure_heuristics(setting_name, apply_fn, time_limit=30):
    m = ac_opf.build_model("small")
    m.hideOutput()
    if apply_fn is not None:
        apply_fn(m)
    m.setParam("limits/time", time_limit)
    t0 = time.time()
    m.optimize()
    return dict(setting=setting_name, status=str(m.getStatus()), nsols=m.getNSols(),
                primal=m.getPrimalbound() if m.getNSols() > 0 else None,
                dual=m.getDualbound(), gap=m.getGap(), nodes=m.getNNodes(),
                time=time.time() - t0)

results = [
    measure_heuristics("既定", None),
    measure_heuristics("heuristics=aggressive", lambda m: m.setHeuristics(SCIP_PARAMSETTING.AGGRESSIVE)),
]
for r in results:
    print(r)

In [ ]:
labels = [r["setting"] for r in results]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("見つけた実行可能解の数", "最良の発電費用(低いほど良い)", "双対境界(高いほど良い)"))
bar_colors = [C["muted"], C["s1"]]
fig.add_trace(go.Bar(x=labels, y=[r["nsols"] for r in results], marker_color=bar_colors,
    text=[r["nsols"] for r in results], textposition="outside", cliponaxis=False,
    showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=labels, y=[r["primal"] for r in results], marker_color=bar_colors,
    text=[f"{r['primal']:.3f}" for r in results], textposition="outside", cliponaxis=False,
    showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=labels, y=[r["dual"] for r in results], marker_color=bar_colors,
    text=[f"{r['dual']:.2f}" for r in results], textposition="outside", cliponaxis=False,
    showlegend=False), row=1, col=3)
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=380,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="ヒューリスティクス強化の効果(30秒予算) — 解の個数は増えるが質は同等、双対境界は悪化",
               x=0.01, font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
show(fig)

**Honest verification result**: Strengthening heuristics increases the number of feasible solutions found (default → aggressive), but **the best generation cost (quality of the primal solution) remains almost unchanged** — the combination of capacitor steps and generation outputs reached a good solution early on even with the default heuristics, and further exploration just added "alternative solutions of similar quality." On the other hand, **the dual bound clearly worsens** (because time spent on heuristics reduces the tightening of the root relaxation by branching). In other words, under this scale and time budget, strengthening heuristics provides no practical value beyond "solution diversity," and is rather counterproductive from the perspective of proving optimality (gap) — demonstrating that the trade-off of "where to spend the time budget" hits much harder in true non-convex MINLPs than in big-M/disjunctive structures.

## Summary

- AC-OPF, which handles voltage, reactive power, and transmission losses unseen in DC approximations, features **true non-convexities** (voltage $\times$ voltage $\times$ trigonometric functions). By designing integers (capacitor steps) to enter the reactive balance linearly, the non-convexity is concentrated on the continuous side $(V, \theta)$ — the spatial branch-and-bound tree in sections 2 and 3 directly corroborates this.
- While all other samples with big-M disjunctive structures (without true non-convexity) were solved at root 1 node, this problem triggers `weak_relaxation` + `dual_stall` even at the minimal scale of 5 buses, and the gap worsens by orders of magnitude with just a roughly threefold increase in buses (14 buses) — **diagnostically, disjunctive structures and true non-convexities belong to entirely different difficulty classes**.
- Because it is truly non-convex, improvements based on proving optimality were unrealistic, and verifying based on the **quality of feasible solutions** was appropriate. Strengthening heuristics increases the number of solutions but does not change their quality, sacrificing the dual bound — this was honestly measured, including the fact that "the improvement doesn't work well."

### Practical Decision-Making Correlated with this Diagnosis

This serves as material for dispatch centers to judge "how far generation costs can be optimized given this system scale and time budget." At scales where proving strict optimality is unrealistic, rather than tuning SCIP settings, **problem partitioning (N-1 scenario reduction, decomposition by voltage bands) and switching to stronger convex relaxations (SOC or QC relaxations)** become more important as future extension challenges (see the "Remaining Issues" in the ac_opf.py docstring).

Related: [Method Guide](../../playbook/index.md) /
Sample Catalog [Energy and Microgrid](../../samples/energy_and_microgrid.md)